# ChromaDB

ChromaDB è un database specificamente progettato per la gestione e la ricerca di embeddings, rappresentazioni numeriche di dati testuali o multimediali utilizzate in applicazioni di intelligenza artificiale. È ottimizzato per casi d'uso come il recupero semantico e l'analisi di similarità, offrendo un'infrastruttura altamente performante per lavorare con dati vettoriali. Grazie alla sua integrazione con librerie di machine learning e modelli pre-addestrati, ChromaDB consente agli sviluppatori di costruire rapidamente sistemi avanzati per la ricerca di informazioni, raccomandazioni, clustering e altre applicazioni basate sulla comprensione semantica. Progettato per essere scalabile e facile da usare, ChromaDB è una soluzione ideale per chi desidera sfruttare embeddings in modo efficiente.

https://docs.trychroma.com/docs/overview/introduction

In [ ]:
%%capture --no-stderr
%pip install chromadb

# Lettura dei documenti

In [ ]:
import os

documents_dir = "resumes"

documents = []
metadatas = []
ids = []

id = 0

for filename in os.listdir(documents_dir):
    if filename.endswith(".txt"):
        with open(os.path.join(documents_dir, filename), "r", encoding="utf-8") as file:
            chunks = file.read().replace('\n', ' ').split('###')
            
            # print(chunks[2])
            
            for chunk in chunks:
                if not chunk.isspace() and not chunk == "":
                    documents.append(chunk)
                    metadatas.append({"source": filename})
                    ids.append(str(id))
                    id += 1

In [ ]:
len(documents)

In [ ]:
metadatas[0]

In [ ]:
documents[0]

In [ ]:
ids[0]

# OpenAI

In [ ]:
%%capture --no-stderr
%pip install openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from chromadb.utils import embedding_functions

load_dotenv()  # Carica le variabili da .env
openai_key = os.getenv("OPENAI_API_KEY", "sk-proj-your-key-here")

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=openai_key,
    model_name="text-embedding-3-small" # "text-embedding-ada-002"
)

In [ ]:
import chromadb
chroma_client = chromadb.Client()

# Using OpenAI embedding function instead of the default model to convert text into embeddings.
collection = chroma_client.get_or_create_collection(
    name="CVs",
    embedding_function=openai_ef
)

In [ ]:
collection.add(
    documents = documents,
    metadatas = metadatas,
    ids = ids
)

# Vector Similarity

In [ ]:
user_question = "mi serve qualcuno per promuovere il mio prodotto"
# user_question = "mi serve qualcuno per mettere in sicurezza il mio sito web"
# user_question = "mi serve qualcuno certificato CEM"

results = collection.query(
    query_texts=user_question,
    n_results = 2
)

results

# Prepariamo il prompt con il contesto

In [ ]:
context = f"CONTESTO: Nome file {results['metadatas'][0][0]['source']} ecco il paragrafo piu' significativo: {results['documents'][0][0]}"

context

In [ ]:
prompt = f"""Dato il seguente contesto {context} rispondi alla domanda dell'utente {user_question}
spiegando che nel file individuato c'e' il profilo piu' adatto.
Argomenta la scelta utilizzando il contenuto del testo individuato nel contesto"""

prompt

# Utilizziamo un modello di linguaggio

In [ ]:
import openai
from openai import OpenAI

client = OpenAI(api_key=openai_key)

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "developer", # prima si chiamava "system" prompt
            "content": "Sei un assistente HR, specializzato nella ricerca di profili professionali",
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(completion.choices[0].message.content)